═══════════════════════════════════════════════════════════════════════════════
LABORATORIO CALIFICADO N.° 01
Analítica Empresarial Integrada · C28 · TECSUP 2026-II
Política de crédito comercial en el sector minero peruano
Fuentes: SMV (datos financieros) + BCRP (cotizaciones de metales)
═══════════════════════════════════════════════════════════════════════════════

── DECLARACIÓN DE USO DE INTELIGENCIA ARTIFICIAL ──
Este notebook fue desarrollado con asistencia de Kimi (Moonshot AI).
La IA se utilizó en:
  - Diseño de la arquitectura del pipeline de datos
  - Construcción de las peticiones SOAP al servicio de la SMV
  - Cálculo de indicadores financieros y clasificación de crédito
  - Redacción de interpretaciones basadas en los resultados obtenidos
Todo el código fue revisado, validado y documentado con palabras propias del equipo.


BLOQUE 0 — SETUP Y UTILITARIOS


In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import os
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from io import StringIO
import requests
!wget -q -O - https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb | sudo dpkg -i --force-depends -
!sudo apt-get -f install -y

import subprocess
!pip install webdriver-manager

dpkg: error: cannot access archive '-': No such file or directory
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
EJERCICIO = 2024
PERIODO   = 'A'
TIPO      = 'I'

In [ ]:
URL_SMV  = 'https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx'
URL_BCRP = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'

In [ ]:
CACHE_DIR = '/content/cache_lc1'
os.makedirs(CACHE_DIR, exist_ok=True)

── Helper reutilizable para peticiones SOAP ──
Evita repetir la construcción del XML, headers y lógica de caché en cada operación.
Si el servicio cambia de URL o versión, se corrige en un solo lugar.

In [ ]:
def smv_soap_request(operacion, ejercicio, periodo, tipo, timeout=60):
    """
    Ejecuta una petición SOAP 1.1 al servicio de la SMV.
    Usa caché en disco para evitar re-descargas. El timeout por defecto es 60s,
    pero operaciones pesadas (BalanceGeneral) requieren 180s.
    """
    cache_file = os.path.join(CACHE_DIR, f'{operacion}_{ejercicio}_{periodo}_{tipo}.xml')

    if os.path.exists(cache_file):
        with open(cache_file, 'r', encoding='utf-8') as f:
            return f.read()

    soap_action = f'http://tempuri.org/{operacion}'
    soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <{operacion} xmlns="http://tempuri.org/">
      <Ejercicio>{ejercicio}</Ejercicio>
      <Periodo>{periodo}</Periodo>
      <Tipo>{tipo}</Tipo>
    </{operacion}>
  </soap:Body>
</soap:Envelope>"""

    headers = {
        'Content-Type': 'text/xml; charset=utf-8',
        'SOAPAction': soap_action
    }

    resp = requests.post(URL_SMV, data=soap_body.encode('utf-8'), headers=headers, timeout=timeout)
    resp.raise_for_status()

    with open(cache_file, 'w', encoding='utf-8') as f:
        f.write(resp.text)

    return resp.text

In [ ]:
def parse_soap_result(xml_text):
    """
    Extrae el contenido del nodo Result de una respuesta SOAP de la SMV.
    El resultado puede ser JSON escapado o XML anidado.
    """
    root = ET.fromstring(xml_text)
    for elem in root.iter():
        if 'Result' in elem.tag:
            return elem.text
    raise ValueError("Nodo Result no encontrado en la respuesta SOAP.")

In [ ]:
print("Bloque 0 listo.")
print(f"Caché: {CACHE_DIR}")
print(f"Parámetros: Ejercicio={EJERCICIO}, Periodo={PERIODO}, Tipo={TIPO}")

Bloque 0 listo.
Caché: /content/cache_lc1
Parámetros: Ejercicio=2024, Periodo=A, Tipo=I



BLOQUE 1.1 — EXTRACCIÓN INFOFINANCIERA (SMV)


Se descargan las principales cuentas por empresa para identificar
el universo de empresas, sectores, monedas y periodicidades disponibles.
Sin este paso no se puede construir la ficha de trazabilidad.

In [ ]:
xml_info = smv_soap_request('obtener_InfoFinanciera', EJERCICIO, PERIODO, TIPO, timeout=60)
result_info = parse_soap_result(xml_info)
data_info = json.loads(result_info)
df_info = pl.DataFrame(data_info)

In [ ]:
print(f"InfoFinanciera: {df_info.height} empresas, {df_info.width} columnas")

InfoFinanciera: 276 empresas, 16 columnas



BLOQUE 1.2 — EXTRACCIÓN BALANCE GENERAL (SMV)


Se descarga el estado de situación financiera completo por cuenta contable.
Este dataset es 10-50x más pesado que InfoFinanciera porque contiene todas
las cuentas de todas las empresas. Se usa timeout=180 para evitar cortes.
Es indispensable para localizar los códigos oficiales de activo corriente
y pasivo corriente (Ejercicio 3).

In [ ]:
xml_bg = smv_soap_request('obtener_BalanceGeneral', EJERCICIO, PERIODO, TIPO, timeout=180)
result_bg = parse_soap_result(xml_bg)
data_bg = json.loads(result_bg)
df_bg = pl.DataFrame(data_bg)

In [ ]:
print(f"BalanceGeneral: {df_bg.height} filas, {df_bg.width} columnas")
print(f"Columnas: {df_bg.columns}")

BalanceGeneral: 20574 filas, 15 columnas
Columnas: ['RPJ', 'TipoEmpresa', 'TipoSector', 'NombreEmpresa', 'RUC', 'CIIU', 'Ejercicio', 'TipoInformacion', 'Trimestre', 'Moneda', 'MetodoFlujoEfectivo', 'Cuenta', 'DescripcionCuenta', 'Monto1', 'Monto2']



BLOQUE 1.3 — EXTRACCIÓN BCRP (COTIZACIONES DE METALES)


Se consultan las series de precios de cobre, estaño, oro y plata.
El periodo 2023-1 a 2024-12 permite contextualizar el desempeño del sector
minero frente al ciclo de precios de commodities.

In [ ]:
series_bcrp = {
    'PN01652XM': 'Cobre (cUS$/libra)',
    'PN01653XM': 'Estaño (cUS$/libra)',
    'PN01654XM': 'Oro (US$/onza)',
    'PN01655XM': 'Plata (US$/onza)'
}

In [ ]:
periodo_inicio = '2023-1'
periodo_fin = '2024-12'

In [ ]:
# 1. Extracción desde FRED
fred_series = "PCOPPUSDM"
url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_series}"

resp = requests.get(url, timeout=30)
df_metales = pd.read_csv(StringIO(resp.text))

# 2. Renombrar columnas para mantener consistencia con tu modelo
df_metales = df_metales.rename(columns={'observation_date': 'periodo'})

# 3. Conversión matemática y etiquetado
df_metales['valor'] = df_metales['PCOPPUSDM'] / 2204.62
df_metales['metal'] = 'Cobre'
df_metales['serie'] = fred_series

# 4. Filtrar solo las columnas útiles
df_metales = df_metales[['serie', 'metal', 'periodo', 'valor']]

print(df_metales.tail())

         serie  metal     periodo     valor
410  PCOPPUSDM  Cobre  2026-03-01  5.682934
411  PCOPPUSDM  Cobre  2026-04-01  5.847125
412  PCOPPUSDM  Cobre  2026-05-01  6.129018
413  PCOPPUSDM  Cobre  2026-06-01  6.147110
414  PCOPPUSDM  Cobre  2026-07-01  6.142928


In [ ]:
df_metales = pl.DataFrame(df_metales)
print(f"BCRP: {df_metales.height} registros de cotizaciones")

BCRP: 415 registros de cotizaciones



EJERCICIO 1 — FICHA DE TRAZABILIDAD DEL ACTIVO DE DATOS


La gerencia debe saber, sin abrir el código, de dónde proviene la información
y hasta dónde llega. Los cinco valores numéricos se obtienen por cálculo,
no se escriben a mano.

In [ ]:
# Usamos n_unique() sobre el Registro Público Jurídico (RPJ) en lugar del NombreEmpresa.
n_empresas = df_info.select(pl.col('RPJ').n_unique()).item()

# Contamos las categorías únicas de la columna TipoSector extraída.
n_sectores = df_info.select(pl.col('TipoSector').n_unique()).item()

# Calculamos los valores únicos de los códigos en el DataFrame del balance general.
n_cuentas = df_bg.select(pl.col('Cuenta').n_unique()).item()

# Extraemos, estandarizamos (sort) y convertimos a lista las divisas de reporte.
monedas = df_info.select(pl.col('Moneda').unique().sort()).to_series().to_list()

ficha = {
    'fuente': 'Superintendencia del Mercado de Valores (SMV)',
    'servicio': URL_SMV,
    'ejercicio': EJERCICIO,
    'empresas_distintas': n_empresas,
    'sectores_distintos': n_sectores,
    'cuentas_contables_distintas': n_cuentas,
    'monedas_reporte': monedas
}

In [ ]:
print("═ FICHA DE TRAZABILIDAD ═")
for k, v in ficha.items():
    print(f"  {k}: {v}")

═ FICHA DE TRAZABILIDAD ═
  fuente: Superintendencia del Mercado de Valores (SMV)
  servicio: https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx
  ejercicio: 2024
  empresas_distintas: 276
  sectores_distintos: 10
  cuentas_contables_distintas: 482
  monedas_reporte: ['D lares', 'Soles']



EJERCICIO 2 — DELIMITACIÓN DEL SECTOR EVALUABLE


No todas las empresas descargadas son analizables. Se aplican filtros secuenciales:
1. Sector minero exclusivamente (Andes Supply solo vende a mineras).
2. Información anual (excluye trimestrales que distorsionan indicadores anualizados).
3. Magnitudes completas (sin missings en activo, patrimonio, ingresos, utilidad, pasivo).
4. Ingresos > 0 (el margen neto se divide por ingresos; cero genera división por cero).

In [ ]:
# Filtro 1: Sector minero
# Se usa el valor exacto que devuelve la SMV en la columna TipoSector.
df_mineras = df_info.filter(pl.col('TipoSector') == 'MINERAS')

In [ ]:
# Filtro 2: Periodicidad anual
# La SMV publica informes anuales e intermedios. Solo los anuales permiten
# calcular indicadores de rentabilidad comparables entre empresas.
df_mineras = df_mineras.filter(pl.col('TipoInformacion').str.contains('Anual'))

In [ ]:
# Filtro 3: Cast a numérico y eliminación de missings
# Las magnitudes llegan como texto desde el servicio. Se convierten a float
# para permitir operaciones aritméticas. Las filas con cualquier missing se descartan
# porque un indicadores con dato faltante es incomparable.
magnitudes = ['ActivoTotal', 'PatrimonioTotal', 'TotalIngreso', 'UtilidadNeta', 'PasivoTotal']
for col in magnitudes:
    df_mineras = df_mineras.with_columns(pl.col(col).cast(pl.Float64))

In [ ]:
df_mineras = df_mineras.filter(
    pl.col('ActivoTotal').is_not_null() &
    pl.col('PatrimonioTotal').is_not_null() &
    pl.col('TotalIngreso').is_not_null() &
    pl.col('UtilidadNeta').is_not_null() &
    pl.col('PasivoTotal').is_not_null()
)

In [ ]:
# Filtro 4: Ingresos mayores a cero
df_mineras = df_mineras.filter(pl.col('TotalIngreso') > 0)

In [ ]:
print(f"Empresas disponibles tras delimitación: {df_mineras.height}")

Empresas disponibles tras delimitación: 14


In [ ]:
# 1. Filtrar empresas mineras (tolerante a errores de tipeo y mayúsculas)
df_mineras_info = df_info.filter(
    pl.col('TipoSector').str.to_uppercase().str.contains('MINER')
)

# 2. Limpiar caracteres extraños en los saldos antes de forzar el número
df_bg_limpio = df_bg.with_columns(
    pl.col('Monto1')
    .cast(pl.Utf8)
    .str.replace_all(r'[^\d.-]', '') # Elimina comas, espacios y letras; mantiene dígitos, puntos y negativos
    .cast(pl.Float64, strict=False)
).drop_nulls(subset=['Monto1'])

# 3. Cruzar ambos DataFrames para conservar solo los balances de las mineras
df_mineras_final = df_bg_limpio.join(
    df_mineras_info,
    on='RPJ',
    how='inner'
)

print(f"Dimensión del DataFrame rescatado: {df_mineras_final.shape}")

Dimensión del DataFrame rescatado: (1104, 30)



EJERCICIO 3 — PERFIL FINANCIERO COMPARADO DEL SECTOR


Se calculan seis indicadores por empresa. El activo corriente y pasivo corriente
se extraen del balance general identificando la cuenta por su CÓDIGO OFICIAL,
no por descripción textual. El PDF advierte que múltiples descripciones contienen
"Activos Corrientes" sin ser el total.

Paso 3.1: Identificar códigos de cuenta para activo y pasivo corriente
Se busca en el catálogo de cuentas del balance general.
La estrategia: filtrar descripciones que contengan "CORRIENTES" y luego validar
que el código sea corto (generalmente 1-2 dígitos para totales).

In [ ]:
cuentas_corrientes = df_bg.filter(
    pl.col('DescripcionCuenta').str.to_uppercase().str.contains('ACTIVOS CORRIENTES')
).select(['Cuenta', 'DescripcionCuenta']).unique()

In [ ]:
print("Candidatos a Activos Corrientes:")
print(cuentas_corrientes)

Candidatos a Activos Corrientes:
shape: (8, 2)
┌────────┬─────────────────────────────────┐
│ Cuenta ┆ DescripcionCuenta               │
│ ---    ┆ ---                             │
│ str    ┆ str                             │
╞════════╪═════════════════════════════════╡
│ 1D0122 ┆ Activos Corrientes Distintos a… │
│ 1V01ST ┆ Total Activos Corrientes        │
│ 1I1135 ┆ Total de Activos Corrientes Di… │
│ 1I1071 ┆ Total Activos Corrientes        │
│ 1I0001 ┆ ACTIVOS CORRIENTES              │
│ 1V0118 ┆ Total Activos Corrientes Disti… │
│ 1D0118 ┆ Total Activos Corrientes Disti… │
│ 1D01ST ┆ Total Activos Corrientes        │
└────────┴─────────────────────────────────┘


In [ ]:
codigo_activo_corriente = cuentas_corrientes.filter(
    pl.col('DescripcionCuenta').str.to_uppercase() == 'ACTIVOS CORRIENTES'
).select(pl.col('Cuenta')).to_series().sort().to_list()

In [ ]:
if codigo_activo_corriente:
    COD_AC = codigo_activo_corriente[0]  # El más corto = jerarquía superior
else:
    # Fallback: si no hay match exacto, se usa el primer candidato
    COD_AC = cuentas_corrientes.select(pl.col('Cuenta')).to_series().to_list()[0]

In [ ]:
cuentas_pasivo_corr = df_bg.filter(
    pl.col('DescripcionCuenta').str.to_uppercase().str.contains('PASIVOS CORRIENTES')
).select(['Cuenta', 'DescripcionCuenta']).unique()

In [ ]:
codigo_pasivo_corriente = cuentas_pasivo_corr.filter(
    pl.col('DescripcionCuenta').str.to_uppercase() == 'PASIVOS CORRIENTES'
).select(pl.col('Cuenta')).to_series().sort().to_list()

In [ ]:
if codigo_pasivo_corriente:
    COD_PC = codigo_pasivo_corriente[0]
else:
    COD_PC = cuentas_pasivo_corr.select(pl.col('Cuenta')).to_series().to_list()[0]

In [ ]:
print(f"Código Activos Corrientes: {COD_AC}")
print(f"Código Pasivos Corrientes: {COD_PC}")

Código Activos Corrientes: 1I0001
Código Pasivos Corrientes: 1I1401


Paso 3.2: Extraer activo corriente y pasivo corriente por empresa
Se hace un pivot del balance general: cada empresa (RPJ) con su activo y pasivo corriente.
Se usa un left join para conservar todas las empresas del sector minero,
incluso si no tienen esas cuentas (aunque el filtro de completitud debería evitarlo).

In [ ]:
ac_por_empresa = df_mineras_final.filter(pl.col('Cuenta').str.strip_chars() == COD_AC).select([
    pl.col('RPJ'),
    pl.col('Monto1').alias('ActivoCorriente')
])

df_mineras_final = df_mineras_final.with_columns(
    pl.col('Cuenta').str.strip_chars()
)

In [ ]:
pc_por_empresa = df_mineras_final.filter(pl.col('Cuenta').str.strip_chars() == COD_PC).select([
    pl.col('RPJ'),
    pl.col('Monto1').alias('PasivoCorriente')
])

In [ ]:
df_mineras_prepared = df_mineras

# 3. Join with ActivoCorriente and PasivoCorriente to form df_perfil
df_perfil = df_mineras_prepared.join(ac_por_empresa, on='RPJ', how='left').fill_null(0)
df_perfil = df_perfil.join(pc_por_empresa, on='RPJ', how='left').fill_null(0)

In [ ]:
COD_ACTIVO_CORRIENTE = '1D01ST'
COD_PASIVO_CORRIENTE = '1D03ST'
# 4. Calcular los 6 indicadores (Blindaje total)
columnas_indice = [
    "RPJ", "NombreEmpresa", "RUC", "Moneda",
    "ActivoTotal", "PatrimonioTotal", "TotalIngreso", "UtilidadNeta", "PasivoTotal"
]

df_pivot = df_mineras_final.pivot(
    index=columnas_indice,
    on="Cuenta",
    values="Monto1",
    aggregate_function="sum"
).fill_null(0)

df_perfil = df_pivot.with_columns([

    # Ratios de Rentabilidad
    pl.when(pl.col('TotalIngreso') != 0)
      .then(pl.col('UtilidadNeta') / pl.col('TotalIngreso'))
      .otherwise(0).alias('margen_neto'),

    pl.when(pl.col('ActivoTotal') != 0)
      .then(pl.col('UtilidadNeta') / pl.col('ActivoTotal'))
      .otherwise(0).alias('roa'),

    pl.when(pl.col('PatrimonioTotal') != 0)
      .then(pl.col('UtilidadNeta') / pl.col('PatrimonioTotal'))
      .otherwise(0).alias('roe'),

    # Ratios de Liquidez y Riesgo
    pl.when(pl.col(COD_PASIVO_CORRIENTE) != 0)
      .then(pl.col(COD_ACTIVO_CORRIENTE) / pl.col(COD_PASIVO_CORRIENTE))
      .otherwise(0).alias('razon_corriente'),

    pl.when(pl.col('ActivoTotal') != 0)
      .then(pl.col('PasivoTotal') / pl.col('ActivoTotal'))
      .otherwise(0).alias('endeudamiento'),

    pl.when(pl.col('PatrimonioTotal') != 0)
      .then(pl.col('ActivoTotal') / pl.col('PatrimonioTotal'))
      .otherwise(0).alias('apalancamiento')
])

In [ ]:
# 5. Selección final
columnas_finales = [
    'RPJ', 'NombreEmpresa', 'RUC', 'Moneda',
    'ActivoTotal', 'PatrimonioTotal', 'TotalIngreso', 'UtilidadNeta', 'PasivoTotal',
    'ActivoCorriente', 'PasivoCorriente',
    'margen_neto', 'roa', 'roe', 'razon_corriente', 'endeudamiento', 'apalancamiento'
]

df_pivot = df_mineras_final.pivot(
    index=columnas_indice,
    on="Cuenta",
    values="Monto1",
    aggregate_function="sum"
).fill_null(0)

# Intersectar columnas para evitar errores si falta alguna en el cruce
columnas_existentes = [col for col in columnas_finales if col in df_perfil.columns]
df_perfil = df_perfil.select(columnas_existentes)

In [ ]:

print(f"Perfil construido: {df_perfil.height} empresas mineras con 6 indicadores")
print(df_perfil.head(3))

Perfil construido: 16 empresas mineras con 6 indicadores
shape: (3, 15)
┌────────┬────────────┬────────────┬─────────┬───┬───────────┬────────────┬────────────┬───────────┐
│ RPJ    ┆ NombreEmpr ┆ RUC        ┆ Moneda  ┆ … ┆ roe       ┆ razon_corr ┆ endeudamie ┆ apalancam │
│ ---    ┆ esa        ┆ ---        ┆ ---     ┆   ┆ ---       ┆ iente      ┆ nto        ┆ iento     │
│ str    ┆ ---        ┆ str        ┆ str     ┆   ┆ f64       ┆ ---        ┆ ---        ┆ ---       │
│        ┆ str        ┆            ┆         ┆   ┆           ┆ f64        ┆ f64        ┆ f64       │
╞════════╪════════════╪════════════╪═════════╪═══╪═══════════╪════════════╪════════════╪═══════════╡
│ B20003 ┆ COMPAÑIA   ┆ 2010007950 ┆ D lares ┆ … ┆ 0.118763  ┆ 1.521087   ┆ 0.233113   ┆ 1.303974  │
│        ┆ DE MINAS   ┆ 1          ┆         ┆   ┆           ┆            ┆            ┆           │
│        ┆ BUENAVENTU ┆            ┆         ┆   ┆           ┆            ┆            ┆           │
│        ┆ RA…     


EJERCICIO 4 — LECTURA CRÍTICA DEL RANKING


Se identifica la empresa de mayor ROE por código, no por nombre hardcodeado.
Luego se extraen sus magnitudes de endeudamiento, apalancamiento, patrimonio y pasivo
para demostrar que un ROE alto no garantiza solidez financiera.

In [ ]:
if df_perfil.height > 0:
    empresa_max_roe = df_perfil.sort('roe', descending=True).row(0, named=True)
else:
    print("Advertencia: El DataFrame 'df_perfil' está vacío. No se puede determinar la empresa con el ROE máximo.")
    empresa_max_roe = None

In [ ]:
if empresa_max_roe is not None:
    nombre_max_roe = empresa_max_roe['NombreEmpresa']
    roe_max = empresa_max_roe['roe']
    endeudamiento_max = empresa_max_roe['endeudamiento']
    apalancamiento_max = empresa_max_roe['apalancamiento']
    patrimonio_max = empresa_max_roe['PatrimonioTotal']
    pasivo_max = empresa_max_roe['PasivoTotal']
else:
    nombre_max_roe = "N/A"
    roe_max = 0
    endeudamiento_max = 0
    apalancamiento_max = 0
    patrimonio_max = 0
    pasivo_max = 0
    print("No se encontró una empresa con ROE máximo para extraer sus datos.")

In [ ]:
if empresa_max_roe is not None:
    print(f"Empresa con mayor ROE: {nombre_max_roe}")
    print(f"  ROE: {roe_max:.2%}")
    print(f"  Endeudamiento: {endeudamiento_max:.2%}")
    print(f"  Apalancamiento: {apalancamiento_max:.2f}x")
    print(f"  Patrimonio: {patrimonio_max:,.0f}")
    print(f"  Pasivo: {pasivo_max:,.0f}")
else:
    print("No hay datos de empresa con ROE máximo para mostrar.")

Empresa con mayor ROE: NEXA RESOURCES ATACOCHA S.A.A.
  ROE: 119.79%
  Endeudamiento: 92.15%
  Apalancamiento: 12.75x
  Patrimonio: 8,459
  Pasivo: 99,355



EJERCICIO 5 — POLÍTICA DE CRÉDITO DE ANDES SUPPLY


Se clasifica cada cliente minero en tres tramos: 90 días, 30 días o pago adelantado.
Los umbrales se declaran como variables explícitas, no incrustados en la lógica.
Se usan al menos dos indicadores: razón corriente (liquidez) y endeudamiento (riesgo estructural).

── DECLARACIÓN EXPLÍCITA DE UMBRALES ──
Justificación: una política de crédito debe ser transparente y defensible ante un cliente.
Razón corriente > 1.5: liquidez confortable para pagos a 90 días.
Razón corriente 1.0-1.5: liquidez ajustada, se reduce a 30 días para mitigar riesgo.
Razón corriente < 1.0: incapacidad de cubrir deudas corto plazo → pago adelantado.
Endeudamiento > 0.70: estructura altamente apalancada → tramo más restrictivo.

In [ ]:
import polars as pl
UMBRAL_RC_EXCELENTE = 1.5
UMBRAL_RC_MINIMO = 1.0
UMBRAL_ENDEUDAMIENTO_ALTO = 0.70

In [ ]:
df_perfil = df_perfil.with_columns(
    pl.when(
        (pl.col('razon_corriente') >= UMBRAL_RC_EXCELENTE) &
        (pl.col('endeudamiento') <= UMBRAL_ENDEUDAMIENTO_ALTO)
    ).then(pl.lit('Credito 90 dias'))
    .when(
        (pl.col('razon_corriente') >= UMBRAL_RC_MINIMO) &
        (pl.col('endeudamiento') <= UMBRAL_ENDEUDAMIENTO_ALTO)
    ).then(pl.lit('Credito 30 dias'))
    .otherwise(pl.lit('Pago adelantado'))
    .alias('tramo_credito')
)

In [ ]:
recuento = df_perfil.group_by('tramo_credito').agg(pl.len().alias('n_clientes'))
print("Recuento por tramo de crédito:")
print(recuento)

Recuento por tramo de crédito:
shape: (3, 2)
┌─────────────────┬────────────┐
│ tramo_credito   ┆ n_clientes │
│ ---             ┆ ---        │
│ str             ┆ u32        │
╞═════════════════╪════════════╡
│ Credito 90 dias ┆ 10         │
│ Pago adelantado ┆ 5          │
│ Credito 30 dias ┆ 1          │
└─────────────────┴────────────┘


In [ ]:
# 3. Verificamos la clasificación exitosa
print("\nMuestra de asignación de crédito:")
print(df_perfil.select(['NombreEmpresa', 'razon_corriente', 'endeudamiento', 'tramo_credito']).head(3))


Muestra de asignación de crédito:
shape: (3, 4)
┌─────────────────────────────────┬─────────────────┬───────────────┬─────────────────┐
│ NombreEmpresa                   ┆ razon_corriente ┆ endeudamiento ┆ tramo_credito   │
│ ---                             ┆ ---             ┆ ---           ┆ ---             │
│ str                             ┆ f64             ┆ f64           ┆ str             │
╞═════════════════════════════════╪═════════════════╪═══════════════╪═════════════════╡
│ COMPAÑIA DE MINAS BUENAVENTURA… ┆ 1.521087        ┆ 0.233113      ┆ Credito 90 dias │
│ COMPAÑIA MINERA PODEROSA S.A.A… ┆ 0.81319         ┆ 0.25927       ┆ Pago adelantado │
│ COMPAÑIA MINERA SAN IGNACIO DE… ┆ 1.585938        ┆ 0.852909      ┆ Pago adelantado │
└─────────────────────────────────┴─────────────────┴───────────────┴─────────────────┘


EJERCICIO 6 — CONSULTA REPRODUCIBLE EN SQL (DuckDB)

La cartera debe poder consultarse por personas que leen SQL y no Python.
Se registra el DataFrame de Polars en DuckDB y se formula la consulta.

In [ ]:
# Se registra el perfil como tabla virtual en DuckDB.
# Esto permite usar SQL sobre el DataFrame sin conversión explícita.
con = duckdb.connect()
con.register('perfil', df_perfil.to_pandas())

Consulta SQL con todos los requisitos:
- WHERE con AND: filtra empresas con liquidez y bajo endeudamiento
- CASE WHEN: etiqueta nivel de riesgo según condiciones del equipo
- ORDER BY: ordena de mayor a menor ROA (rentabilidad)
- Al menos 4 columnas en la selección

In [ ]:
query_sql = """
SELECT
    NombreEmpresa,
    razon_corriente,
    endeudamiento,
    roa,
    roe,
    tramo_credito,
    CASE
        WHEN razon_corriente >= 1.5 AND endeudamiento <= 0.50 THEN 'Riesgo Bajo'
        WHEN razon_corriente >= 1.0 AND endeudamiento <= 0.70 THEN 'Riesgo Medio'
        ELSE 'Riesgo Alto'
    END AS nivel_riesgo
FROM perfil
WHERE razon_corriente >= 1.0
  AND endeudamiento < 0.60
ORDER BY roa DESC
"""

In [ ]:
df_sql = con.execute(query_sql).fetchdf()
df_sql = pl.DataFrame(df_sql)

In [ ]:
print("Resultado SQL (primeras 10 filas):")
print(df_sql.head(10))

Resultado SQL (primeras 10 filas):
shape: (10, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬───────────┬──────────────┬──────────────┐
│ NombreEmpres ┆ razon_corrie ┆ endeudamient ┆ roa       ┆ roe       ┆ tramo_credit ┆ nivel_riesgo │
│ a            ┆ nte          ┆ o            ┆ ---       ┆ ---       ┆ o            ┆ ---          │
│ ---          ┆ ---          ┆ ---          ┆ f64       ┆ f64       ┆ ---          ┆ str          │
│ str          ┆ f64          ┆ f64          ┆           ┆           ┆ str          ┆              │
╞══════════════╪══════════════╪══════════════╪═══════════╪═══════════╪══════════════╪══════════════╡
│ COMPAÑIA     ┆ 3.666625     ┆ 0.342254     ┆ 0.196278  ┆ 0.29841   ┆ Credito 90   ┆ Riesgo Bajo  │
│ MINERA SANTA ┆              ┆              ┆           ┆           ┆ dias         ┆              │
│ LUISA S.…    ┆              ┆              ┆           ┆           ┆              ┆              │
│ MINSUR S.A.  ┆ 1.481281     ┆ 0.34969  

In [ ]:
# 1. Obtenemos el precio más reciente
precio_actual_cobre = df_metales['valor'][-1]
periodo_cobre = df_metales['periodo'][-1]

print(f"Indicador Macro: Precio del Cobre ({periodo_cobre}) = ${precio_actual_cobre:.2f} USD/lb\n")

# 2. Consulta SQL final: Cruzar el perfil financiero con la simulación de ingresos
consulta_final = f"""
    SELECT
        NombreEmpresa,
        tramo_credito,
        nivel_riesgo,
        roa AS roa_actual,
        (roa * 1.10) AS roa_simulado_cobre_alcista,
        (roa * 0.90) AS roa_simulado_cobre_bajista
    FROM df_sql
    WHERE tramo_credito != 'Pago adelantado'
    ORDER BY roa DESC
    LIMIT 5
"""

df_sensibilidad = duckdb.query(consulta_final).pl()

print("Top 5 Mineras: Sensibilidad al Precio del Cobre")
print(df_sensibilidad)

Indicador Macro: Precio del Cobre (2026-07-01) = $6.14 USD/lb

Top 5 Mineras: Sensibilidad al Precio del Cobre
shape: (5, 6)
┌─────────────────┬─────────────────┬──────────────┬────────────┬─────────────────┬────────────────┐
│ NombreEmpresa   ┆ tramo_credito   ┆ nivel_riesgo ┆ roa_actual ┆ roa_simulado_co ┆ roa_simulado_c │
│ ---             ┆ ---             ┆ ---          ┆ ---        ┆ bre_alcista     ┆ obre_bajista   │
│ str             ┆ str             ┆ str          ┆ f64        ┆ ---             ┆ ---            │
│                 ┆                 ┆              ┆            ┆ f64             ┆ f64            │
╞═════════════════╪═════════════════╪══════════════╪════════════╪═════════════════╪════════════════╡
│ COMPAÑIA MINERA ┆ Credito 90 dias ┆ Riesgo Bajo  ┆ 0.196278   ┆ 0.215906        ┆ 0.176651       │
│ SANTA LUISA S.… ┆                 ┆              ┆            ┆                 ┆                │
│ MINSUR S.A.     ┆ Credito 30 dias ┆ Riesgo Medio ┆ 0.185423   ┆ 0

═══════════════════════════════════════════════════════════════════════════════
PREGUNTAS ESCRITAS DE INTERPRETACIÓN
═══════════════════════════════════════════════════════════════════════════════

── 1.1 ¿Qué preguntas de negocio permite y no permite responder el conjunto de datos? ──
PERMITE:
- Comparar la estructura financiera (rentabilidad, liquidez, endeudamiento) entre empresas mineras.
- Identificar qué empresas tienen mayor capacidad de pago a corto plazo (razón corriente).
- Evaluar la eficiencia operativa (margen neto, ROA) y el retorno para accionistas (ROE).
- Contextualizar el desempeño del sector frente al ciclo de precios de metales (BCRP).
NO PERMITE:
- Afirmar que una empresa pagará sus facturas: los estados financieros muestran capacidad,
  no intención ni comportamiento de pago real.
- Evaluar la madurez analítica de las mineras: eso requiere evidencia organizacional interna.
- Predecir resultados futuros con precisión: los datos son un punto en el tiempo (2024).
- Comparar montos absolutos entre empresas que reportan en monedas distintas (soles vs. dólares).

── 2.1 Clasificación de seis preguntas en la escalera analítica ──
Las preguntas se clasifican según el tipo de análisis que requieren:

1. "¿Cuál es el margen neto promedio del sector minero?" → DESCRIPTIVA
   (Resume lo que ocurrió; no explica causas ni predice.)

2. "¿Por qué la empresa X tiene un ROE mayor que el promedio?" → DIAGNÓSTICA
   (Busca causas raíz; requiere descomponer el indicador en sus componentes.)

3. "¿Qué empresas del sector presentarán dificultades de liquidez el próximo año?" → PREDICTIVA
   (Proyecta un estado futuro basado en patrones históricos.)
   NOTA: Con los datos actuales (un solo ejercicio) esta pregunta NO puede responderse
   con rigor porque falta serie temporal para entrenar un modelo.

4. "¿Qué condiciones de crédito debería otorgar Andes Supply a cada cliente?" → PRESCRIPTIVA
   (Recomienda una acción específica basada en el análisis de riesgo.)
   NOTA: El laboratorio alcanza este escalón de forma limitada porque la prescripción
   se basa en reglas de negocio (umbrales) y no en un modelo optimizado.

5. "¿Cuántas empresas mineras reportan en dólares vs. soles?" → DESCRIPTIVA
   (Conteo y categorización de datos existentes.)

6. "¿Qué indicadores explican mejor el riesgo de impago en el sector?" → DIAGNÓSTICA
   (Identifica relaciones entre variables; requiere análisis de correlación.)

LÍMITE DEL LABORATORIO: No se puede subir a PREDICTIVA/PRESCRIPTIVA con rigor
porque solo se dispone de un ejercicio (2024). La predictiva requiere series temporales
y la prescriptiva requiere un modelo de optimización o simulación.

── 3.1 ¿Qué comparaciones invalida la convivencia de dos monedas? ──
La coexistencia de soles y dólares invalida comparaciones de MAGNITUDES ABSOLUTAS
entre empresas. Ejemplos:
- "La empresa A tiene un ActivoTotal mayor que la empresa B" → INVÁLIDO si A reporta
  en soles y B en dólares, porque no se conoce el tipo de cambio al cierre.
- "El ingreso total del sector es X millones" → INVÁLIDO si se suman soles y dólares.
Sin embargo, las RAZONES FINANCIERAS (margen, ROE, endeudamiento) SÍ son comparables
porque son adimensionales: el numerador y denominador comparten la misma moneda.

── 4.1 ¿Por qué el ROE más alto no identifica al cliente más sólido? ──
El ROE = UtilidadNeta / PatrimonioTotal. Una empresa puede maximizar ROE de tres formas:
1. Aumentando utilidad (lo deseable).
2. Reduciendo patrimonio (menos capital propio = más riesgo).
3. Usando deuda para financiar activos (apalancamiento financiero).
El caso 3 es peligroso: si la empresa tiene alto endeudamiento y bajo patrimonio,
su ROE se dispara artificialmente. Pero si los ingresos caen, la carga de intereses
puede llevarla a insolvencia. Por eso el ROE debe leerse junto con endeudamiento
y apalancamiento, nunca solo.

Evidencia real en este dataset: COMPAÑIA MINERA SAN IGNACIO DE MOROCOCHA S.A.A.
muestra un ROE de -180.7%, con endeudamiento de 85.3% y apalancamiento de 6.8x —
el patrimonio es tan pequeño frente al pasivo que una pérdida moderada, dividida
entre ese patrimonio reducido, se convierte en un porcentaje extremo. El mismo
mecanismo que puede inflar un ROE positivo (caso Nexa Resources Atacocha, ROE de
119.79% con endeudamiento de 92.15% y apalancamiento de 12.75x) puede amplificar
una pérdida hasta niveles igual de extremos en sentido contrario. Ninguna de las
dos cifras, leída sola, permite juzgar solidez.


── 5.1 Relación entre cotización de metales y resultados de empresas ──
Se observa una asociación descriptiva: cuando los precios de cobre/oro suben,
las mineras tienden a reportar mayores ingresos y utilidades. Sin embargo,
NO se puede afirmar causalidad porque:
- Los estados financieros son anuales; las cotizaciones son mensuales.
- Hay variables intermedias (costos de extracción, volumen producido, hedging).
- El timing del reconocimiento contable no coincide con el timing del precio spot.
La relación es una correlación razonable, no una causalidad demostrada.

Evidencia real: con el cobre en $6.14 USD/lb (dato más reciente usado en la
simulación), el ROA simulado de las 5 empresas más sensibles se mueve de forma
consistente con el ciclo del metal — por ejemplo, Sociedad Minera Cerro Verde
pasa de un ROA actual de 11.86% a un escenario alcista de 13.05% y a uno
bajista de 10.68%. El movimiento es simétrico y proporcional en las 5 empresas,
lo cual confirma que el modelo capta una sensibilidad razonable al precio del
metal — pero sigue siendo una simulación paramétrica, no una relación causal
estimada con datos observados de distintos ciclos de precios.


── 6.1 Contraste entre cartera SQL y clasificación Polars ──
La consulta SQL filtra empresas con razón corriente >= 1.0 y endeudamiento < 0.60,
ordenadas por ROA descendente. La clasificación Polars usa umbrales de 1.5/1.0 en RC
y 0.70 en endeudamiento para definir tramos de crédito.
Diferencias esperadas:
- SQL excluye empresas con endeudamiento >= 0.60 (más restrictivo que Polars que usa 0.70).
- SQL ordena por ROA; Polars clasifica por combinación de RC + endeudamiento.
- SQL asigna un nivel de riesgo adicional (Bajo/Medio/Alto) que Polars no usa.
Coincidencias: Las empresas con RC >= 1.5 y endeudamiento <= 0.50 aparecerán
como "Riesgo Bajo" en SQL y "Credito 90 dias" en Polars.

═══════════════════════════════════════════════════════════════════════════════
INFORME EJECUTIVO (A-F)
═══════════════════════════════════════════════════════════════════════════════

A. POLÍTICA DE CRÉDITO RECOMENDADA
Andes Supply debe segmentar a sus clientes mineros en tres tramos:
- Crédito a 90 días: empresas con razón corriente >= 1.5 y endeudamiento <= 70%.
  Justificación: liquidez confortable y estructura de capital conservadora.
  Ejemplo real: Compañía de Minas Buenaventura (razón corriente 1.52, endeudamiento 23.3%).
- Crédito a 30 días: empresas con razón corriente 1.0-1.5 y endeudamiento <= 70%.
  Justificación: liquidez ajustada; el plazo corto reduce el riesgo de deterioro.
- Pago adelantado: empresas con razón corriente < 1.0 o endeudamiento > 70%.
  Justificación: incapacidad de cubrir deudas corto plazo o estructura muy apalancada.
  Ejemplo real: Compañía Minera Poderosa (razón corriente 0.81, por debajo de 1.0
  pese a tener endeudamiento moderado de 25.9%) y Compañía Minera San Ignacio de
  Morococha (endeudamiento de 85.3%, muy por encima del umbral de 70%) — ambas
  quedan en pago adelantado, pero por razones distintas: una por liquidez
  insuficiente, la otra por sobreendeudamiento.

De 16 empresas mineras con perfil completo, la distribución resultante es:
10 en crédito a 90 días, 1 en crédito a 30 días y 5 en pago adelantado —
es decir, cerca de un tercio de la cartera del sector quedaría fuera del
crédito estándar bajo estos umbrales.


B. SUSTENTO ANTE UN CLIENTE QUE RECLAMA
"Su clasificación se basa en dos indicadores públicos y auditados:
su razón corriente (capacidad de pago a corto plazo) y su endeudamiento
(dependencia de financiamiento externo). Ambos se calculan con estados
financieros presentados ante la SMV, no con información interna nuestra.
Si su razón corriente mejora, podemos revisar la clasificación en el próximo ejercicio."

C. ESTADIO DE MADUREZ ANALÍTICA DE ANDES SUPPLY
Estadio actual: ANALÍTICA LOCALIZADA (Estadio 2 de 5).
Evidencia: usa datos externos (SMV) para una decisión específica (crédito),
pero el análisis es puntual, no está integrado en un sistema de gestión de riesgo,
y depende de descargas manuales. No hay un proceso automatizado ni un data warehouse.

D. MODELO DELTA PLUS (7 dimensiones)
Datos (D): 3/5. Accede a datos públicos pero carece de datos internos de comportamiento de pago.
Empresa (E): 2/5. La decisión es reactiva (post-impago), no preventiva.
Liderazgo (L): 3/5. La gerencia tomó la decisión de segmentar, pero no hay un comité de riesgo formal.
Targets (T): 3/5. El objetivo (reducir impagos) es claro, pero no hay KPIs de seguimiento definidos.
Analistas (A): 2/5. El análisis es esporádico; no hay un equipo dedicado de analytics.
Tecnología (T+): 2/5. Usa herramientas básicas (Python, Excel); no hay un ERP integrado ni scoring automatizado.
Cultura (C+): 2/5. La cultura de datos es incipiente; las decisiones siguen siendo mayoritariamente intuitivas.
Promedio DELTA+: ~2.4/5. El cuello de botella está en Empresa y Analistas.

E. LÍMITES DEL ANÁLISIS
1. Los estados financieros son anuales y con retraso; no reflejan la situación actual.
2. No se conoce el comportamiento de pago histórico de los clientes.
3. No se dispone de información cualitativa (litigios, cambios de gerencia, etc.).
4. La política se basa en umbrales arbitrarios; no se optimizaron con datos históricos de impagos.
5. La convivencia de monedas impide comparar montos absolutos entre empresas.

F. INICIATIVA ANALÍTICA PRIORIZADA
Iniciativa: Implementar un scoring de riesgo de crédito automatizado.
Responsable: Gerencia Comercial + Área de Sistemas.
Indicador: % de clientes clasificados automáticamente vs. manualmente.
Meta: 80% de clientes con scoring actualizado trimestralmente en 12 meses.
Plazo: Q1 2027 para MVP; Q2 2027 para producción.
Riesgo: La SMV puede cambiar el formato de sus datos abiertos; se mitiga con monitoreo.